# Task 1 : Unknown function discovering scientist agent
Entirely in basic Python and without using any agent framework or large-language-model API, a simple autonomous “scientist agent” whose task is to identify an unknown mathematical function by interacting with an environment through a limited set of tools. At the start of each run, the environment secretly chooses one function from a small family, for example a linear function $ y= ax+ b $ or a quadratic function $ y= ax^2 + bx+ c $, with coefficients hidden from the agent;observations may contain noise so that the environment returns
$ y= f(x) + ε $, where $ ε $ is a small random error. 

In [1]:
import random
import math

In [ ]:
# ---------------------------------------------------------------------------
# ENVIRONMENT
# ---------------------------------------------------------------------------
class Environment:
    """
    Hidden ground truth. Secretly linear or quadratic, with optional
    Gaussian noise: y = f(x) + eps, eps ~ N(0, sigma^2).
 
    x_range is treated as a KNOWN constraint (like a known instrument range) -- the agent may query anywhere in it,
    it just doesn't know which family f belongs to or its coefficients.
    """
 
    def __init__(self, kind=None, x_range=(-10, 10), sigma=0.0, seed=None):
        rng = random.Random(seed)
        self.kind = kind if kind is not None else rng.choice(["linear", "quadratic"])
        self.x_range = x_range
        self.sigma = sigma
        self._rng = rng
 
        if self.kind == "linear":
            self.a = 0.0
            self.b = rng.uniform(-5, 5)
            self.c = rng.uniform(-10, 10)  # reuse b/c naming: y = b*x + c
        else:
            self.a = rng.uniform(-2, 2)
            self.b = rng.uniform(-5, 5)
            self.c = rng.uniform(-10, 10)
 
    def true_function(self, x):
        return self.a * x * x + self.b * x + self.c
 
    def experiment(self, x):
        """TOOL: experiment(x_t) -> noisy y_t."""
        noise = self._rng.gauss(0, self.sigma) if self.sigma > 0 else 0.0
        return self.true_function(x) + noise
 
    def sample_test_points(self, n=50, seed=None):
        """Held-out points for FINAL evaluation only. Agent never sees these."""
        rng = random.Random(seed)
        lo, hi = self.x_range
        xs = [rng.uniform(lo, hi) for _ in range(n)]
        ys = [self.true_function(x) for x in xs]  # noiseless ground truth for scoring
        return xs, ys

In [3]:
# ---------------------------------------------------------------------------
# MEMORY
# ---------------------------------------------------------------------------
class Memory:
    def __init__(self):
        self.observations = []
 
    def add(self, x, y):
        self.observations.append((x, y))
 
    def xs(self):
        return [p[0] for p in self.observations]
 
    def ys(self):
        return [p[1] for p in self.observations]
 
    def __len__(self):
        return len(self.observations)

In [ ]:
# ---------------------------------------------------------------------------
# FITTING TOOLS (plain arithmetic, no ML libraries)
# ---------------------------------------------------------------------------
def fit_linear(xs, ys):
    """
    Mean-centered OLS for y = a*x + b (numerically cleaner than raw sums, -- good practice).
    """
    n = len(xs)
    if n < 2:
        raise ValueError("Need >= 2 points to fit a line.")
    mean_x = sum(xs) / n
    mean_y = sum(ys) / n
    num = sum((x - mean_x) * (y - mean_y) for x, y in zip(xs, ys))
    den = sum((x - mean_x) ** 2 for x in xs)
    if abs(den) < 1e-12:
        raise ValueError("Degenerate design: all x-values (nearly) identical.")
    a = num / den
    b = mean_y - a * mean_x
    return a, b
 
 
def fit_quadratic(xs, ys):
    """
    OLS for y = a*x^2 + b*x + c via the 3x3 normal equations, solved with
    Gaussian elimination + partial pivoting (numerically-safe pattern).
    """
    n = len(xs)
    if n < 3:
        raise ValueError("Need >= 3 points to fit a quadratic.")
 
    sum_x = sum(xs)
    sum_x2 = sum(x * x for x in xs)
    sum_x3 = sum(x ** 3 for x in xs)
    sum_x4 = sum(x ** 4 for x in xs)
    sum_y = sum(ys)
    sum_xy = sum(x * y for x, y in zip(xs, ys))
    sum_x2y = sum(x * x * y for x, y in zip(xs, ys))
 
    # Augmented matrix [A | rhs] for [a, b, c]
    A = [
        [sum_x4, sum_x3, sum_x2, sum_x2y],
        [sum_x3, sum_x2, sum_x, sum_xy],
        [sum_x2, sum_x, n, sum_y],
    ]
 
    for i in range(3):
        pivot = max(range(i, 3), key=lambda r: abs(A[r][i]))
        if abs(A[pivot][i]) < 1e-12:
            raise ValueError("Degenerate design: cannot solve for quadratic coefficients.")
        A[i], A[pivot] = A[pivot], A[i]
        for j in range(3):
            if i != j:
                ratio = A[j][i] / A[i][i]
                for k in range(4):
                    A[j][k] -= ratio * A[i][k]
 
    a = A[0][3] / A[0][0]
    b = A[1][3] / A[1][1]
    c = A[2][3] / A[2][2]
    return a, b, c
 
 
def predict_linear(a, b, x):
    return a * x + b
 
 
def predict_quadratic(a, b, c, x):
    return a * x * x + b * x + c
 
 
def rss(preds, ys):
    return sum((p - y) ** 2 for p, y in zip(preds, ys))
 
 
def validate(predict_fn, xs_test, ys_test):
    """TOOL: validate(hypothesis, held-out points) -> MSE."""
    n = len(xs_test)
    sq_err = sum((predict_fn(x) - y) ** 2 for x, y in zip(xs_test, ys_test))
    return sq_err / n

In [ ]:
# ---------------------------------------------------------------------------
# POLICY: extremes-first bootstrap, then disagreement-maximizing queries
# ---------------------------------------------------------------------------
def choose_next_x(memory, x_range):
    """
    n < 3   : extremes-first bootstrap (D-optimal for a single linear fit;
              also the minimum needed before a quadratic fit exists at all).
    n >= 3  : query x* = argmax_{x in [lo,hi]} |g(x)|, where
              g(x) = quad_pred(x) - lin_pred(x) is the two-hypotheses'
              disagreement function. g is itself quadratic in x, so the
              maximum of |g| over a closed interval is attained either at
              the interior vertex of g (where g'(x)=0) or at one of the
              two endpoints -- we check all three candidates explicitly
              (vertex alone is NOT sufficient).
    """
    lo, hi = x_range
    n = len(memory)
 
    if n == 0:
        return lo
    if n == 1:
        return hi
    if n == 2:
        # third point needed to make quadratic fit possible; avoid
        # collinearity-with-extremes degeneracy by using a interior probe
        return (lo + hi) / 2.0
 
    xs, ys = memory.xs(), memory.ys()
    try:
        a_l, b_l = fit_linear(xs, ys)
        a_q, b_q, c_q = fit_quadratic(xs, ys)
    except ValueError:
        # fits not yet possible/degenerate -- fall back to extremes
        return lo if n % 2 == 0 else hi
 
    # g(x) = a_q x^2 + (b_q - a_l) x + (c_q - b_l)
    ga, gb = a_q, (b_q - a_l)
    # gc = c_q - b_l   (unused: doesn't affect argmax location)
 
    candidates = [lo, hi]
    if abs(ga) > 1e-12:
        x_vertex = -gb / (2 * ga)
        if lo <= x_vertex <= hi:
            candidates.append(x_vertex)
 
    def g_abs(x):
        return abs(predict_quadratic(a_q, b_q, c_q, x) - predict_linear(a_l, b_l, x))
 
    return max(candidates, key=g_abs)

In [ ]:
# ---------------------------------------------------------------------------
# NESTED-MODEL SEQUENTIAL LIKELIHOOD-RATIO STOPPING RULE
# ---------------------------------------------------------------------------
CHI2_CRIT_DF1_ALPHA05 = 3.841  # standard chi-square(1) critical value, alpha=0.05
 
def lrt_statistic(xs, ys):
    """
    D = n * ln(RSS_linear / RSS_quadratic)   (== 2 * log-likelihood-ratio
    under Gaussian noise with MLE variance -- see module docstring).
    Returns (D, a_l, b_l, a_q, b_q, c_q) or None if fits aren't possible.
    """
    n = len(xs)
    try:
        a_l, b_l = fit_linear(xs, ys)
        a_q, b_q, c_q = fit_quadratic(xs, ys)
    except ValueError:
        return None
 
    rss_l = rss([predict_linear(a_l, b_l, x) for x in xs], ys)
    rss_q = rss([predict_quadratic(a_q, b_q, c_q, x) for x in xs], ys)
 
    if rss_q < 1e-12:
        rss_q = 1e-12  # guard against perfect-fit division
    if rss_l < 1e-12:
        rss_l = 1e-12
 
    D = n * math.log(rss_l / rss_q)
    return D, a_l, b_l, a_q, b_q, c_q
 
 
def should_stop(state, memory, stability_window=4, accept_linear_thresh=0.5,
                 min_points_for_decision=6):
    """
    Layered stopping rule:
      Layer 1 - hard stop on budget exhaustion.
      Layer 2 - reject linear (accept quadratic) once D exceeds the
                chi-square(1) critical value for `stability_window`
                consecutive queries -- the statistically grounded half.
      Layer 3 - accept linear once D stays below accept_linear_thresh
                for `stability_window` consecutive queries, AND enough
                points have been collected to have given quadratic a
                fair chance -- the heuristic half (see module docstring
                limitation note).
    """
    if state.budget_remaining <= 0:
        return True, "budget_exhausted", None
 
    n = len(memory)
    if n < min_points_for_decision:
        return False, "insufficient_data", None
 
    result = lrt_statistic(memory.xs(), memory.ys())
    if result is None:
        return False, "degenerate_fit_continue", None
    D, a_l, b_l, a_q, b_q, c_q = result
 
    state.D_history.append(D)
    recent = state.D_history[-stability_window:]
    if len(recent) < stability_window:
        return False, "gathering_stability_window", None
 
    if all(d > CHI2_CRIT_DF1_ALPHA05 for d in recent):
        return True, "quadratic_accepted", ("quadratic", a_q, b_q, c_q)
 
    if all(d < accept_linear_thresh for d in recent):
        return True, "linear_accepted", ("linear", a_l, b_l)
 
    return False, "continue", None

In [ ]:
# ---------------------------------------------------------------------------
# AGENT STATE + LOOP
# ---------------------------------------------------------------------------
class AgentState:
    def __init__(self, budget):
        self.budget_total = budget
        self.budget_remaining = budget
        self.D_history = []
 
 
def run_agent(env, budget=40, verbose=False):
    memory = Memory()
    state = AgentState(budget)
    decision = None
    reason = None
 
    while True:
        stop, reason, decision = should_stop(state, memory)
        if stop:
            break
 
        x_t = choose_next_x(memory, env.x_range)
        y_t = env.experiment(x_t)
        state.budget_remaining -= 1
        memory.add(x_t, y_t)
 
        if verbose:
            print(f"  query #{len(memory)}: x={x_t:8.3f} -> y={y_t:8.3f}")
 
    # Budget exhausted without a confident decision: fall back to
    # lower-AIC model, explicitly flagged as low-confidence
    if decision is None:
        result = lrt_statistic(memory.xs(), memory.ys()) if len(memory) >= 3 else None
        if result is not None:
            D, a_l, b_l, a_q, b_q, c_q = result
            n = len(memory)
            aic_l = n * math.log(rss([predict_linear(a_l, b_l, x) for x in memory.xs()], memory.ys()) / n) + 2 * 2
            aic_q = n * math.log(rss([predict_quadratic(a_q, b_q, c_q, x) for x in memory.xs()], memory.ys()) / n) + 2 * 3
            if aic_q < aic_l:
                decision = ("quadratic", a_q, b_q, c_q)
            else:
                decision = ("linear", a_l, b_l)
            reason = "budget_exhausted_fallback_aic"
        else:
            reason = "budget_exhausted_insufficient_data"
 
    return state, memory, decision, reason

In [9]:
# ---------------------------------------------------------------------------
# OBJECTIVE
# ---------------------------------------------------------------------------
def evaluate(env, memory, decision, lam=0.01):
    xs_test, ys_test = env.sample_test_points(n=50, seed=123)
    if decision is None:
        mse = float("inf")
    elif decision[0] == "linear":
        _, a, b = decision
        mse = validate(lambda x: predict_linear(a, b, x), xs_test, ys_test)
    else:
        _, a, b, c = decision
        mse = validate(lambda x: predict_quadratic(a, b, c, x), xs_test, ys_test)
 
    n_queries = len(memory)
    J = mse + lam * n_queries
    return {"mse_test": mse, "n_queries": n_queries, "J": J}

In [10]:
# ---------------------------------------------------------------------------
# DEMO HARNESS
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    print("=== Stage 4: Full Scientist Agent (noisy, active policy, LRT stopping) ===\n")
 
    random.seed(0)
    for trial in range(6):
        kind = "linear" if trial % 2 == 0 else "quadratic"
        env = Environment(kind=kind, sigma=0.8, seed=trial)
        true_str = (f"y = {env.b:.3f}x + {env.c:.3f}" if kind == "linear"
                    else f"y = {env.a:.3f}x^2 + {env.b:.3f}x + {env.c:.3f}")
        print(f"[trial {trial}] true ({kind}): {true_str}")
 
        state, memory, decision, reason = run_agent(env, budget=40)
        if decision is None:
            print(f"  -> no decision reached ({reason})\n")
            continue
 
        if decision[0] == "linear":
            _, a, b = decision
            found_str = f"y = {a:.3f}x + {b:.3f}"
        else:
            _, a, b, c = decision
            found_str = f"y = {a:.3f}x^2 + {b:.3f}x + {c:.3f}"
 
        metrics = evaluate(env, memory, decision)
        print(f"  decided: {decision[0]}  ({reason})")
        print(f"  found:   {found_str}")
        print(f"  queries used: {metrics['n_queries']}  MSE_test: {metrics['mse_test']:.4f}  J: {metrics['J']:.4f}\n")

=== Stage 4: Full Scientist Agent (noisy, active policy, LRT stopping) ===

[trial 0] true (linear): y = 3.444x + 5.159
  decided: linear  (linear_accepted)
  found:   y = 3.452x + 5.149
  queries used: 15  MSE_test: 0.0024  J: 0.1524

[trial 1] true (quadratic): y = -1.463x^2 + 3.474x + 5.275
  decided: quadratic  (quadratic_accepted)
  found:   y = -1.457x^2 + 3.555x + 5.084
  queries used: 9  MSE_test: 0.1946  J: 0.2846

[trial 2] true (linear): y = 4.560x + 8.957
  decided: linear  (linear_accepted)
  found:   y = 4.548x + 8.630
  queries used: 10  MSE_test: 0.1033  J: 0.2033

[trial 3] true (quadratic): y = -1.048x^2 + 0.442x + -2.601
  decided: quadratic  (quadratic_accepted)
  found:   y = -1.058x^2 + 0.468x + -2.027
  queries used: 9  MSE_test: 0.1815  J: 0.2715

[trial 4] true (linear): y = -2.640x + -7.937
  decided: linear  (linear_accepted)
  found:   y = -2.634x + -7.873
  queries used: 12  MSE_test: 0.0042  J: 0.1242

[trial 5] true (quadratic): y = 0.492x^2 + 2.418x + 5.